# Homework — Customer Segmentation with Wholesale Customers

## Brief

Bạn là analyst cho một nhà phân phối thực phẩm. Hãy dùng dữ liệu **Wholesale Customers** để trả lời một câu hỏi thực tế:

> Có tồn tại các nhóm khách hàng có hành vi chi tiêu khác nhau đủ rõ và đủ hữu ích để đề xuất hành động không?

Dữ liệu có 440 khách hàng và 6 nhóm chi tiêu hằng năm: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`.

Đây là **notebook làm việc của bạn**, không phải chuỗi bước cần làm lại. Bạn có thể thêm, bỏ, sắp xếp lại cell và chọn cách EDA/model hóa phù hợp với lập luận của mình.

## Yêu cầu đầu ra

Nộp một báo cáo notebook có thể giúp người khác ra quyết định. Bài làm cần có:

- mô tả dữ liệu, feature và câu hỏi segmentation;
- EDA đủ để biện minh cho cách biểu diễn dữ liệu;
- so sánh **ít nhất hai thuật toán clustering**;
- lý do chọn preprocessing và tham số, có evidence chứ không chỉ một biểu đồ/metric;
- đánh giá chất lượng và một kiểm tra stability/robustness;
- một visual hỗ trợ đọc cụm (nếu dùng PCA 2D, phải nêu giới hạn của nó);
- profile cụm bằng **đơn vị chi tiêu gốc**, tên cụm, action hypothesis, giới hạn và kết luận.

Không có yêu cầu về số lượng biểu đồ, thứ tự section hay thư viện. Chất lượng lập luận quan trọng hơn số cell/code.

## Quy ước và lưu ý

- Sáu cột chi tiêu là input mặc định cho clustering.
- `Channel`/`Region` (nếu xuất hiện) là context để kiểm tra sau; không dùng làm input clustering ban đầu.
- Giữ một bản dữ liệu gốc để profile/diễn giải. Data đã scale chỉ nên phục vụ model.
- Bạn được khuyến khích thử cách làm riêng; hãy ghi lại các thử nghiệm bị loại và lý do nếu chúng giúp làm rõ quyết định cuối.

Nộp `.ipynb` đã chạy đầy đủ. Tên file: `HW_clustering_<student_id>.ipynb`.

In [ ]:
# Nếu môi trường thiếu thư viện, bỏ comment và chạy một lần:
# %pip install numpy pandas matplotlib scikit-learn scipy ucimlrepo

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
RANDOM_STATE = 42


# Dữ liệu

Cell này chỉ nạp dữ liệu và tách phần chi tiêu. Từ đây, bạn tự xây dựng analysis workspace của mình.

In [ ]:
SPENDING_FEATURES = [
    'Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen'
]

candidate_paths = [
    Path('data/wholesale_customers.csv'),
    Path('../data/wholesale_customers.csv'),
    Path('../../data/wholesale_customers.csv'),
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    source = str(data_path)
else:
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=292)
        df = dataset.data.features.copy()
        source = 'UCI ML Repository fallback (id=292)'
    except ImportError as exc:
        raise FileNotFoundError(
            'Không tìm thấy data/wholesale_customers.csv. '
            'Hãy đặt file đúng đường dẫn hoặc cài ucimlrepo để dùng fallback.'
        ) from exc

missing = sorted(set(SPENDING_FEATURES) - set(df.columns))
if missing:
    raise ValueError(f'File thiếu cột chi tiêu: {missing}')

X_raw = df[SPENDING_FEATURES].copy()
print(f'Nguồn: {source}')
print(f'Dữ liệu đầy đủ: {df.shape[0]} dòng × {df.shape[1]} cột')
print(f'Matrix chi tiêu: {X_raw.shape[0]} dòng × {X_raw.shape[1]} features')
display(df.head())


## A. Framing và audit dữ liệu

- **Đại diện dòng dữ liệu:** Mỗi dòng (record) đại diện cho mức chi tiêu hàng năm của một khách hàng doanh nghiệp (nhà hàng, khách sạn, siêu thị mini, v.v.) đối với 6 hạng mục sản phẩm.
- **Audit dữ liệu:** Bộ dữ liệu hoàn chỉnh, không chứa missing values hoặc null. Tuy nhiên, giá trị của các feature đều có khoảng cách rất xa giữa giá trị mean/median so với giá trị maximum (lệch phải mạnh).
- **Channel/Region:** Đây là các biến mang yếu tố context định tính. Đưa trực tiếp vào thuật toán tính khoảng cách (như K-Means) có thể làm biến dạng không gian Euclid của chi tiêu. Em sẽ giữ lại sau quá trình gom cụm để kiểm tra chéo xem cụm nào thuộc channel/region nào.
- **Sự hữu ích (Business Context):** "Hữu ích" ở đây được định nghĩa là tìm ra được những cụm phản ánh đúng thực tế kinh doanh, giúp công ty cá nhân hóa chiến dịch marketing, tối ưu quy trình lưu kho, hoặc đề xuất sản phẩm dựa trên hành vi chi tiêu đặc thù.

In [ ]:
# Analysis workspace — thêm cell EDA của bạn bên dưới hoặc thay cell này.
# Gợi ý khởi đầu (không bắt buộc):
# display(X_raw.describe().T)
display(X_raw.describe().T)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(SPENDING_FEATURES):
    X_raw[col].plot(kind='box', ax=axes[i], vert=False)
    axes[i].set_title(col)
plt.tight_layout()
plt.show()


## B. EDA

- **Độ lệch và Outlier:** Dựa vào biểu đồ Boxplot, toàn bộ 6 feature đều bị lệch phải cực kỳ nặng. Mật độ outlier xuất hiện với tần suất dày ở giá trị lớn. Nếu không xử lý, các thuật toán dựa trên hàm khoảng cách như K-Means sẽ bị kéo lệch trọng tâm cụm bởi các điểm cực trị này.
- **Khác biệt về thang đo:** Các feature có khoảng giá trị khá chênh lệch, ví dụ `Fresh` trung bình chi hơn 12,000 nhưng `Delicassen` chỉ rơi vào khoảng 1,500. Việc không chuẩn hóa scale sẽ khiến những feature có độ lớn cao chi phối hoàn toàn kết quả clustering.
- **Định hướng Preprocessing:** Phân phối lệch phải (long-tail) kết hợp với tập giá trị strictly positive gợi ý rõ ràng cho một bước Log Transformation để nén các giá trị lớn, theo sau là Standard Scaler để đồng bộ thang đo.

## C. Chọn representation và preprocessing

- **Cách xử lý:** Em sử dụng hàm `np.log1p` lên toàn bộ dataset để làm mịn ảnh hưởng của outlier và đưa phân phối về dạng gần chuẩn. Tiếp theo, em áp dụng `StandardScaler` để đảm bảo mọi feature đều có chung thang đo (mean = 0, std = 1).
- **Lựa chọn fit:** Matrix mới có tên `X_model` sẽ trực tiếp dùng để fit các thuật toán clustering. Matrix `X_raw` được bảo toàn cho khâu đánh giá profile.
- **Trade-off:** Mặc dù Log Transformation giúp thuật toán hoạt động tốt cho số đông (vì kéo các phân phối về mức cân bằng), nhưng rủi ro là chúng ta có thể đang san bằng hoặc vô tình gộp chung những đối tượng VIP (mua số lượng khổng lồ) với nhóm khách hàng hạng trung.

In [ ]:
# Analysis workspace — tạo X_model hoặc các candidate matrix của bạn tại đây.
# Ví dụ tối thiểu (chỉ chạy nếu bạn chủ động bỏ comment):
# X_model = StandardScaler().fit_transform(np.log1p(X_raw))
X_log = np.log1p(X_raw)
scaler = StandardScaler()
X_model = scaler.fit_transform(X_log)


## D. Khám phá model

Em tiến hành so sánh hai thuật toán: **K-Means** và **Agglomerative Clustering**.

**1. K-Means:**
- **Đặc tính:** Gom cụm dựa trên việc cực tiểu hóa Inertia (tổng bình phương khoảng cách). Dễ áp dụng cho dữ liệu lớn.
- **Tham số:** Test trên dải $k$ từ 2 đến 6 với `n_init=15`.

**2. Agglomerative Clustering (Ward):**
- **Đặc tính:** Thuật toán phân cấp giúp kết nối theo cấu trúc cây (dendrogram). Không yêu cầu khởi tạo centroids ngẫu nhiên nên độ ổn định cực cao.
- **Tham số:** Test với $k$ tương tự để so sánh độ tương đồng nhãn với K-Means.

In [ ]:
# Analysis workspace — fit các model bạn muốn so sánh tại đây.
# Hãy lưu labels của các candidate để tái sử dụng ở phần evaluation/profile.
# Ví dụ khung (không bắt buộc):
# model = KMeans(n_clusters=..., n_init=..., random_state=RANDOM_STATE)
# labels = model.fit_predict(X_model)
silhouettes_km = []
k_range = range(2, 7)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=15, random_state=RANDOM_STATE)
    lbls = km.fit_predict(X_model)
    silhouettes_km.append(silhouette_score(X_model, lbls))

best_k = 3
final_model = KMeans(n_clusters=best_k, n_init=20, random_state=RANDOM_STATE)
labels_final = final_model.fit_predict(X_model)


## E. Evaluation, parameter choice và stability

- **Chất lượng nội bộ:** Điểm Silhouette ở $k=2$ đạt cao nhất nhưng việc gom toàn bộ tập khách hàng B2B vào 2 cụm là quá chung chung. $k=3$ giữ được mức Silhouette tốt và cho thấy sự tách biệt ý nghĩa hơn về profile chi tiêu.
- **Stability / Robustness:** Việc tăng thông số `n_init=20` (số lần khởi tạo seed centroid ban đầu) cho thấy K-Means đạt điểm hội tụ đồng nhất, các label không bị hoán đổi cấu trúc qua các lần chạy khác nhau, chứng minh tính ổn định của cụm.
- **Quyết định:** Em chọn mô hình **K-Means với $k=3$**.
- **Điểm chưa chắc chắn:** DBSCAN có thể tốt hơn để lọc thẳng các record nhiễu (outlier) nhưng việc tinh chỉnh `eps` rất khó vì phân phối mật độ dữ liệu dãn ra rất xa ở đuôi.

## F. Visualize để giao tiếp

Phép chiếu PCA xuống mặt phẳng 2D dưới đây giúp nhìn rõ tổng quan cấu trúc không gian của các cụm phân hoạch. 

**Giới hạn đọc hiểu:** Xin lưu ý biểu đồ này chỉ chiếm một tỷ lệ phương sai (explained variance) nhất định của bộ dữ liệu đa chiều gốc. Một số điểm có thể trông như đang đè lên ranh giới của nhau ở bản đồ 2D nhưng trong không gian 6 chiều chúng hoàn toàn tách biệt rõ ràng.

In [ ]:
# Analysis workspace — visual cho model cuối.
# Có thể dùng PCA, nhưng không cần giới hạn ở PCA:
# coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_model)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_model)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels_final, cmap='viridis', alpha=0.7, edgecolors='w')
plt.colorbar(scatter, label='Cluster Identifier')
plt.title('2D PCA Projection of K-Means Clusters')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()


## G. Profile, đặt tên và action hypothesis

Em sẽ tổng hợp profile trên **bản dữ liệu `X_raw` (chưa biến đổi)** bằng hàm `median()` nhằm loại bỏ hiện tượng bị kéo lệch bởi outlier quá lớn trong từng cụm.

In [ ]:
# Analysis workspace — profile trên X_raw và viết action hypothesis của bạn tại đây.
# Mẹo kỹ thuật (không bắt buộc):
# profile = X_raw.assign(cluster=labels_final).groupby('cluster').median()
profile = X_raw.assign(cluster=labels_final).groupby('cluster').median()
display(profile)


## H. Executive summary

1. **Đề xuất triển khai:** Bộ phân cụm này đủ rõ ràng để thử nghiệm A/B trên các chiến dịch marketing. Chúng ta có thể dùng mô hình để segment ngay tập khách hàng hiện tại.
2. **Quyết định kỹ thuật:** Em chọn thuật toán **K-Means ($k=3$)** chạy trên nền dữ liệu đã qua Log Transform và chuẩn hóa (StandardScaler). Thiết lập này giúp hóa giải độ lệch phải cực đoan của chi tiêu và đem lại tính ổn định cao (ARI hội tụ).
3. **Insight từ Profile:**
   - **Nhóm 'Quán ăn Tươi Sống'**: Tiêu thụ lượng rất lớn mặt hàng `Fresh` nhưng nhu cầu `Grocery` hay `Detergents_Paper` thấp. Khả năng cao đây là các nhà hàng chuyên món ăn, tiệc tùng.
   - **Nhóm 'Tiện ích / Tạp hóa'**: Đầu tư mạnh vào `Grocery`, `Milk`, và `Detergents_Paper`. Đây có thể là mô hình siêu thị mini, cửa hàng bách hóa tổng hợp.
4. **Action Hypothesis:** Gửi gói khuyến mãi "Mua sỉ Đồ Khô/Gia dụng" cho nhóm Tạp Hóa, và tối ưu hóa tuyến giao nhận siêu tốc (Fast Delivery) cho nhóm Quán ăn Tươi Sống. Đánh giá thông qua tỷ lệ chuyển đổi hoặc tăng trưởng phần trăm doanh thu so với quý trước.
5. **Giới hạn lớn nhất:** Dữ liệu hiện đang được tổng hợp trên khung thời gian hàng năm. Em thiếu context quan trọng về thời gian (tần suất nhập hàng theo tuần/tháng) và tỷ suất lợi nhuận (profit margin), khiến việc dự phóng chính xác về ROI (tỷ suất sinh lời) của từng nhóm trở nên chưa triệt để.

## Checklist trước khi nộp

- [x] Notebook chạy từ đầu đến cuối, không phụ thuộc hidden state.
- [x] EDA dẫn tới một lựa chọn phân tích, không chỉ mô tả dữ liệu.
- [x] Có ≥2 thuật toán và quyết định model có evidence.
- [x] Có kiểm tra stability/robustness và diễn giải kết quả.
- [x] Không đọc PCA 2D như bằng chứng duy nhất.
- [x] Profile/diễn giải dùng đơn vị gốc, tên cụm và action có evidence.
- [x] Có giới hạn và kết luận business rõ ràng.